# 04 HNSC_LASSO_derivation

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import KFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
frozen = pd.read_csv("FROZEN_candidate_pool.csv")
full_coverage = pd.read_csv("full_candidate_pool_final_coverage.csv")
frozen_genes = frozen["Gene"].tolist()

sub = full_coverage[full_coverage["Gene"].isin(frozen_genes)]
gene_to_col = {}
for _, r in sub.iterrows():
    gene_to_col[r["Gene"]] = r["Gene"] if r["TCGA_HNSC_Match"] == "Direct" else r["TCGA_HNSC_Alias_Used"]

print("Candidate pool genes mapped to TCGA-HNSC columns:", len(gene_to_col))

In [ ]:
tcga_cols = list(gene_to_col.values())
usecols = ["Patient ID", "Sample ID", "Overall Survival (Months)", "Overall Survival Status"] + tcga_cols
df = pd.read_csv("clinical_rna_fire_combine.csv", usecols=usecols, engine="python")

df = df[df["Sample ID"].str.endswith("-01")].copy()

for c in tcga_cols:
    df[c] = np.log2(df[c].astype(float) + 1)

df["OS_months"] = df["Overall Survival (Months)"]
df["Event"] = df["Overall Survival Status"].astype(str).str.startswith("1").astype(int)
df = df.dropna(subset=["OS_months", "Event"])
df = df[df["OS_months"] > 0].reset_index(drop=True)

print("Derivation cohort: N =", df.shape[0], "| Events =", int(df["Event"].sum()))

In [ ]:
z_cols = {}
for gene, col in gene_to_col.items():
    z_cols[gene] = (df[col] - df[col].mean()) / df[col].std()

X = pd.concat(z_cols, axis=1)
X.columns = list(gene_to_col.keys())
y = Surv.from_dataframe("Event", "OS_months", df)

print("Design matrix:", X.shape)
X.to_csv("X_zscored.csv", index=False)
df[["Patient ID", "OS_months", "Event"]].to_csv("survival_data.csv", index=False)

In [ ]:
full_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01, n_alphas=100)
full_model.fit(X.values, y)
alphas = full_model.alphas_
print(f"Alpha path: {len(alphas)} values, from {alphas.max():.5f} down to {alphas.min():.5f}")

In [ ]:
N_FOLDS = 10
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_scores = np.full((N_FOLDS, len(alphas)), np.nan)

for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.values[train_idx], X.values[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    try:
        fold_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=alphas, alpha_min_ratio=0.01)
        fold_model.fit(X_train, y_train)
        for a_idx, alpha in enumerate(alphas):
            try:
                risk_scores = fold_model.predict(X_test, alpha=alpha)
                cv_scores[fold_idx, a_idx] = concordance_index_censored(
                    y_test["Event"], y_test["OS_months"], risk_scores)[0]
            except Exception:
                continue
    except Exception as e:
        print(f"Fold {fold_idx} failed: {e}")
    print(f"Fold {fold_idx+1}/{N_FOLDS} done")

In [ ]:
mean_cv = np.nanmean(cv_scores, axis=0)
se_cv = np.nanstd(cv_scores, axis=0) / np.sqrt(np.sum(~np.isnan(cv_scores), axis=0))

best_idx = np.nanargmax(mean_cv)
alpha_min = alphas[best_idx]
best_score = mean_cv[best_idx]
best_se = se_cv[best_idx]
print(f"Best CV C-index: {best_score:.4f} +/- {best_se:.4f} at alpha_min = {alpha_min:.5f}")

threshold = best_score - best_se
candidates_1se = [(a, m) for a, m in zip(alphas, mean_cv) if not np.isnan(m) and m >= threshold]
alpha_1se = max(candidates_1se, key=lambda x: x[0])[0]
print(f"alpha_1se (sparsest within 1 SE) = {alpha_1se:.5f}")

cv_results = pd.DataFrame({"alpha": alphas, "mean_c_index": mean_cv, "se_c_index": se_cv})
cv_results.to_csv("cv_results.csv", index=False)

In [ ]:
def fit_and_report(alpha, label):
    model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[alpha])
    model.fit(X.values, y)
    coefs = model.coef_.flatten()
    genes = X.columns.tolist()
    selected = [(g, c) for g, c in zip(genes, coefs) if c != 0]
    selected.sort(key=lambda x: -abs(x[1]))

    risk_scores = model.predict(X.values, alpha=alpha)
    in_sample_c = concordance_index_censored(y["Event"], y["OS_months"], risk_scores)[0]

    result_df = pd.DataFrame(selected, columns=["Gene", "Coefficient"])
    result_df["HR_per_SD"] = np.exp(result_df["Coefficient"])
    result_df.to_csv(f"HNSC_hypoxia_signature_{label}.csv", index=False)

    print(f"{label}: {len(selected)} genes selected | in-sample C-index = {in_sample_c:.3f}")
    return result_df, in_sample_c

df_min, cidx_min = fit_and_report(alpha_min, "lambda_min")
df_1se, cidx_1se = fit_and_report(alpha_1se, "lambda_1se")

In [ ]:
print("=== lambda.min signature ===")
df_min

In [ ]:
print("=== lambda.1se signature ===")
df_1se

In [ ]:
print("Cross-validated C-index (honest, out-of-sample estimate):", f"{best_score:.3f} +/- {best_se:.3f}")
print()
print(f"lambda.min: {df_min.shape[0]} genes, in-sample C-index = {cidx_min:.3f}")
print(f"lambda.1se: {df_1se.shape[0]} genes, in-sample C-index = {cidx_1se:.3f}")
print()
overlap = set(df_min['Gene']) & set(df_1se['Gene'])
print(f"Genes shared between both models: {len(overlap)} / {df_1se.shape[0]} of the lambda.1se genes")